# Analyse of Population development in Zurich 1910 - 2025 by district and  neighborhood

Source: https://www.stadt-zuerich.ch/de/politik-und-verwaltung/statistik-und-daten/daten/bevoelkerung/bestand-und-entwicklung/aktuelle-bevoelkerung-und-entwicklung.html#entwicklung_nachquartier 

In [36]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Set renderer for notebook display
pio.renderers.default = "notebook"

# load the data
df = pd.read_csv("data/KOT103T1031_Bevoelkerungsentwicklung_nach-Stadtquartier-und-Stadtkreis.csv")

In [37]:
df.drop(columns=["1950"], inplace=True)

In [39]:
# Reshape data to long format for plotting
df_long = df.melt(id_vars=['Kreis', 'Quartier'], 
                   var_name='Year', 
                   value_name='Population')
df_long['Year'] = df_long['Year'].astype(int)

# Plot 1: Aggregated line graph by Kreis (City Districts)
df_by_kreis = df_long.groupby(['Kreis', 'Year'])['Population'].sum().reset_index()

# Determine which Kreise grew between 1960 and 2025
growth_comparison = df_by_kreis.pivot(index='Kreis', columns='Year', values='Population')
growing_kreise = growth_comparison[growth_comparison[2025] > growth_comparison[1960]].index.tolist()

# Separate data for growing and non-growing Kreise
df_growing = df_by_kreis[df_by_kreis['Kreis'].isin(growing_kreise)]
df_not_growing = df_by_kreis[~df_by_kreis['Kreis'].isin(growing_kreise)]

# Create figure with two traces (without markers)
fig1 = px.line(df_not_growing, 
               x='Year', 
               y='Population', 
               color='Kreis',
               title='Population Growth by City District (Kreis)',
               labels={'Population': 'Total Population', 'Kreis': 'District'})

# Add thicker lines for growing Kreise
fig1.add_traces(
    px.line(df_growing, 
            x='Year', 
            y='Population', 
            color='Kreis').data
)

# Update line widths and styles - thicker for growing Kreise, dashed for non-growing
for i, trace in enumerate(fig1.data):
    if any(int(trace.name.split()[-1]) == kreis for kreis in growing_kreise if trace.name.split()[-1].isdigit()):
        trace.line.width = 3
    else:
        trace.line.width = 2
        trace.line.dash = 'dash'

# Add direct labels at the end of each line with colors matching the line colors
max_year = df_by_kreis['Year'].max()
for kreis in sorted(df_by_kreis['Kreis'].unique()):
    last_row = df_by_kreis[(df_by_kreis['Kreis'] == kreis) & (df_by_kreis['Year'] == max_year)]
    if not last_row.empty:
        y_value = last_row['Population'].values[0]
        # Find the corresponding trace to get its color
        trace_color = None
        for trace in fig1.data:
            if f"Kreis {kreis}" in trace.name or str(kreis) in trace.name:
                trace_color = trace.line.color
                break
        
        fig1.add_annotation(
            x=max_year,
            y=y_value,
            text=f"Kreis {kreis}",
            showarrow=False,
            xanchor='left',
            xshift=5,
            font=dict(color=trace_color) if trace_color else dict()
        )

fig1.update_layout(
    showlegend=False,
    xaxis_range=[df_by_kreis['Year'].min() - 1, 2030]
)

pio.show(fig1)

In [33]:
# Plot 2: Line graph for each Quartier (Neighborhood)
fig2 = px.line(df_long, 
               x='Year', 
               y='Population', 
               color='Quartier',
               title='Population Growth by Neighborhood (Quartier)',
               labels={'Population': 'Population', 'Quartier': 'Neighborhood'})

pio.show(fig2)